# New Corpus Experiments

Experiments using the quelle behavioural-projections corpus (7,304 prompts).

**Corpus tiers:**
| Tier | Files | ~Count |
|------|-------|--------|
| Routine bulk | `benchmarks_mmlu.jsonl`, `benchmarks_gsm8k.jsonl` | 33% |
| Diversity | `semantic_diversity.jsonl`, `periphery_probes.jsonl` | 33% |
| Targeted perturbation | `perturbation_families.jsonl` | 33% |

**Experiments:**
- **Exp 5** — Full layer sweep (all 24 Pythia layers), 20 corpus-sampled probes, α=1 & 1000, CFG=25. Primary: convergence layer via seed variance. D12 test: does variance profile shape differ for periphery vs. benchmark probes?
- **Exp 6** — Perturbation family CLIP coherence. 4 families × 6 surface variants at L23, α=1. Do rephrase/negation/register variants cluster with their base in CLIP space?
- **Exp 7** — LPIPS sensitivity at CFG=25 (deferred; reuses Exp 5 images once generated).

**Key architectural point (D12):** Periphery probes are in-corpus → d_act ≈ 0 for all probe categories by construction. The corpus-distance confound that plagued Exp 1–3 is eliminated by design.

In [ ]:
import json
import random
import subprocess
from pathlib import Path

import numpy as np

# ── Paths ────────────────────────────────────────────────────────────────────
SLICER_DIR = Path("/home/user/slicer")
NOTEBOOKS_DIR = SLICER_DIR / "notebooks"
RESULTS_DIR = SLICER_DIR / "experiment_results_new_corpus"

# ── Corpus source ─────────────────────────────────────────────────────────────
# Set CORPUS_LOCAL to a Path once repos are integrated; leave None to fetch from GitHub.
CORPUS_LOCAL: Path | None = None
CORPUS_URL_BASE = (
    "https://raw.githubusercontent.com/leonorae/quelle/"
    "claude/review-and-scaffold-1XMqg/"
    "experiments/behavioral-projections/prompts"
)

# ── Reproducibility ───────────────────────────────────────────────────────────
PROBE_SEED = 42

## 1. Load corpus files

In [ ]:
import urllib.request

def load_jsonl(filename: str) -> list[dict]:
    """Load a JSONL file from local path or GitHub raw URL."""
    if CORPUS_LOCAL is not None:
        path = Path(CORPUS_LOCAL) / filename
        with open(path) as f:
            return [json.loads(line) for line in f if line.strip()]
    url = f"{CORPUS_URL_BASE}/{filename}"
    with urllib.request.urlopen(url) as r:
        return [json.loads(line) for line in r.read().decode().splitlines() if line.strip()]

print("Loading corpus files...")
mmlu         = load_jsonl("benchmarks_mmlu.jsonl")
gsm8k        = load_jsonl("benchmarks_gsm8k.jsonl")
semantic     = load_jsonl("semantic_diversity.jsonl")
periphery    = load_jsonl("periphery_probes.jsonl")
perturbation = load_jsonl("perturbation_families.jsonl")
corpus_full  = load_jsonl("corpus_full.jsonl")

print(f"  MMLU:          {len(mmlu):,}")
print(f"  GSM8K:         {len(gsm8k):,}")
print(f"  Semantic:      {len(semantic):,}")
print(f"  Periphery:     {len(periphery):,}")
print(f"  Perturbation:  {len(perturbation):,}")
print(f"  Full corpus:   {len(corpus_full):,}")

## 2. Sample Exp 5 probes

Source-stratified: 4 per category, 20 probes total.

| Category | Source | N | Selection |
|----------|--------|---|-----------|
| `benchmark_mmlu` | `benchmarks_mmlu.jsonl` | 4 | Diverse subcategories |
| `benchmark_gsm8k` | `benchmarks_gsm8k.jsonl` | 4 | Random |
| `semantic` | `semantic_diversity.jsonl` | 4 | Random |
| `periphery` | `periphery_probes.jsonl` | 4 | One per subcategory: malformed, nonsense, mixed_language, domain_outlier |
| `perturbation_base` | `perturbation_families.jsonl` | 4 | Base variants from 4 different group_ids |

In [ ]:
rng = random.Random(PROBE_SEED)

# ── MMLU: pick 4 entries spanning different subcategories ─────────────────────
mmlu_by_sub: dict[str, list] = {}
for r in mmlu:
    mmlu_by_sub.setdefault(r.get("subcategory", "unknown"), []).append(r)
# Pick one entry from each of 4 subcategories sampled without replacement
chosen_subs = rng.sample(sorted(mmlu_by_sub.keys()), min(4, len(mmlu_by_sub)))
benchmark_mmlu_probes = [rng.choice(mmlu_by_sub[s]) for s in chosen_subs]

# ── GSM8K: 4 random ───────────────────────────────────────────────────────────
benchmark_gsm8k_probes = rng.sample(gsm8k, 4)

# ── Semantic: 4 random ────────────────────────────────────────────────────────
semantic_probes = rng.sample(semantic, 4)

# ── Periphery: one per target subcategory ─────────────────────────────────────
PERIPHERY_SUBCATS = ["malformed", "nonsense", "mixed_language", "domain_outlier"]
periph_by_sub: dict[str, list] = {}
for r in periphery:
    periph_by_sub.setdefault(r.get("subcategory", "unknown"), []).append(r)
periphery_probes = [rng.choice(periph_by_sub[s]) for s in PERIPHERY_SUBCATS if s in periph_by_sub]
if len(periphery_probes) < 4:
    print(f"Warning: only {len(periphery_probes)} periphery subcategories found: {list(periph_by_sub.keys())}")

# ── Perturbation base: 4 different group_ids ──────────────────────────────────
base_entries = [r for r in perturbation if r.get("perturbation_type") == "base"]
exp5_base_entries = rng.sample(base_entries, 4)
EXP5_GROUP_IDS = [r["group_id"] for r in exp5_base_entries]
perturbation_base_probes = exp5_base_entries

# ── Assemble Exp 5 probe_texts dict ──────────────────────────────────────────
def texts(entries): return [e["text"] for e in entries]

EXP5_PROBE_TEXTS = {
    "benchmark_mmlu":    texts(benchmark_mmlu_probes),
    "benchmark_gsm8k":   texts(benchmark_gsm8k_probes),
    "semantic":          texts(semantic_probes),
    "periphery":         texts(periphery_probes),
    "perturbation_base": texts(perturbation_base_probes),
}

total = sum(len(v) for v in EXP5_PROBE_TEXTS.values())
print(f"Exp 5 probe set: {total} probes")
for cat, ts in EXP5_PROBE_TEXTS.items():
    print(f"  {cat}: {len(ts)}")
    for t in ts:
        print(f"    • {t[:80]}")

## 3. Sample Exp 6 probes (perturbation families)

Same 4 group_ids as Exp 5, all 6 surface variants each.  
Exp 5 already generates the `base` variants at all layers; Exp 6 adds the 5 non-base variants at L23 only.

In [ ]:
# Pull all 6 variants for each of the 4 selected group_ids
pert_by_group: dict[str, list] = {}
for r in perturbation:
    gid = r.get("group_id")
    if gid in EXP5_GROUP_IDS:
        pert_by_group.setdefault(gid, []).append(r)

# Canonical variant order
VARIANT_ORDER = ["base", "rephrase", "context_added", "authority_bias", "negation", "register_shift"]

EXP6_PROBE_TEXTS: dict[str, list[str]] = {}
for gid in EXP5_GROUP_IDS:
    variants = {r["perturbation_type"]: r["text"] for r in pert_by_group.get(gid, [])}
    ordered = [variants[v] for v in VARIANT_ORDER if v in variants]
    EXP6_PROBE_TEXTS[gid] = ordered

total6 = sum(len(v) for v in EXP6_PROBE_TEXTS.values())
print(f"Exp 6 probe set: {total6} probes across {len(EXP6_PROBE_TEXTS)} families")
for gid, ts in EXP6_PROBE_TEXTS.items():
    print(f"  {gid}: {len(ts)} variants")
    for i, (vtype, t) in enumerate(zip(VARIANT_ORDER, ts)):
        print(f"    [{vtype}] {t[:70]}")

## 4. Write corpus training texts

Extracts the `text` field from `corpus_full.jsonl` and writes one text per line.  
Passed to `run_experiment.py` via `--training_texts_file`.

In [ ]:
CORPUS_TXT = SLICER_DIR / "corpus_new.txt"

lines = [r["text"] for r in corpus_full if r.get("text", "").strip()]
CORPUS_TXT.write_text("\n".join(lines))
print(f"Wrote {len(lines):,} training texts → {CORPUS_TXT}")

## 5. Write experiment configs

In [ ]:
BASE_CONFIG = {
    "models": {
        "llm": "EleutherAI/pythia-410m",
        "sd": "sd-legacy/stable-diffusion-v1-5",
        "clip_model": "ViT-L-14",
        "clip_pretrained": "openai",
    },
    "output": {
        "base_dir": str(RESULTS_DIR),
        "image_format": "png",
    },
}

# ── Exp 5: full layer sweep ───────────────────────────────────────────────────
exp5_config = {
    **BASE_CONFIG,
    "_comment": "Exp 5: full layer sweep, corpus-sampled probes, D12 convergence-layer test.",
    "projections": {
        "types": ["per_layer"],
        "alpha_values": [1, 1000],
        "training_data_size": 7304,
    },
    "probe_texts": EXP5_PROBE_TEXTS,
    "layers": list(range(24)),           # all 24 Pythia layers
    "cfg_values": [25.0],
    "seeds": list(range(16)),
    "track_lpips": False,               # seed variance is primary; LPIPS deferred to Exp 7
}

# ── Exp 6: perturbation families, L23 only ────────────────────────────────────
# Reuses the trained projection from the same output dir — only adds new generate entries.
exp6_config = {
    **BASE_CONFIG,
    "_comment": "Exp 6: perturbation family coherence. All 6 variants at L23, alpha=1 only.",
    "projections": {
        "types": ["per_layer"],
        "alpha_values": [1],
        "training_data_size": 7304,
    },
    "probe_texts": EXP6_PROBE_TEXTS,
    "layers": [23],
    "cfg_values": [25.0],
    "seeds": list(range(16)),
    "track_lpips": False,
}

exp5_cfg_path = SLICER_DIR / "experiment_config_exp5_layersweep.json"
exp6_cfg_path = SLICER_DIR / "experiment_config_exp6_perturbation.json"

exp5_cfg_path.write_text(json.dumps(exp5_config, indent=2))
exp6_cfg_path.write_text(json.dumps(exp6_config, indent=2))
print(f"Wrote: {exp5_cfg_path}")
print(f"Wrote: {exp6_cfg_path}")

## 6. Run Exp 5

Train once on the full corpus, then generate + analyze.  
**Runtime:** train ≈ overnight (per DECISIONS.md Phase 0); generate ≈ 20 probes × 24 layers × 2 alpha × 16 seeds = 15,360 images.

Run phases separately so you can inspect after each step.

In [ ]:
# Phase: train  (run overnight)
!cd /home/user/slicer && python run_experiment.py \
    --config experiment_config_exp5_layersweep.json \
    --training_texts_file corpus_new.txt \
    --phase train

In [ ]:
# Phase: generate
!cd /home/user/slicer && python run_experiment.py \
    --config experiment_config_exp5_layersweep.json \
    --phase generate

In [ ]:
# Phase: grids + analyze  (grids compose PNGs; analyze produces convergence_summary.json)
!cd /home/user/slicer && python run_experiment.py \
    --config experiment_config_exp5_layersweep.json \
    --phase grids analyze

## 7. Run Exp 6

Reuses the trained projection from Exp 5 (same `output.base_dir`).  
Only the `generate` phase is needed — it adds the non-base variant images without touching existing results.

In [ ]:
# Exp 6: generate perturbation variants at L23  (no train needed — reuses Exp 5 projection)
!cd /home/user/slicer && python run_experiment.py \
    --config experiment_config_exp6_perturbation.json \
    --phase generate

## 8. Load manifest

In [ ]:
import sys
sys.path.insert(0, str(SLICER_DIR))

manifest_path = RESULTS_DIR / "manifest.json"
with open(manifest_path) as f:
    manifest = json.load(f)

seed_variance  = manifest.get("seed_variance", {})
clip_vecs      = manifest.get("probe_clip_vectors", {})
corpus_dists   = manifest.get("probe_corpus_distances", {})
probe_stats    = manifest.get("probe_text_stats", {})

print(f"seed_variance entries:    {len(seed_variance)}")
print(f"probe_clip_vectors keys:  {list(clip_vecs.keys())}")
print(f"probe_text_stats slugs:   {len(probe_stats)}")

## 9. Exp 5 analysis — convergence layer

Layer vs. `mean_pixel_var`, one line per probe, coloured by corpus category.  
The layer of minimum variance is the convergence layer for that probe.

**D12 prediction:** if effective rank is context-dependent, periphery probes (malformed/nonsense) should show a different variance profile shape than benchmark probes — either a sharper minimum at a different layer, or a flatter trajectory indicating the projection is less constrained.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from diffusion_microscope.experiment import _slug

# Build reverse map: slug → category label
SLUG_TO_CAT: dict[str, str] = {}
for cat, probe_list in EXP5_PROBE_TEXTS.items():
    for t in probe_list:
        SLUG_TO_CAT[_slug(t)] = cat

CAT_COLORS = {
    "benchmark_mmlu":    "#2196F3",   # blue
    "benchmark_gsm8k":   "#03A9F4",   # light blue
    "semantic":          "#4CAF50",   # green
    "periphery":         "#FF5722",   # deep orange
    "perturbation_base": "#9C27B0",   # purple
}

# Parse seed_variance: key = "{proj_key}/{slug}/L{NNNN}/CFG{v}"
# Group by (proj_key, cfg) → slug → {layer: mean_pixel_var}
from collections import defaultdict
sv_groups: dict = defaultdict(lambda: defaultdict(dict))

for key, rec in seed_variance.items():
    parts = key.split("/")
    if len(parts) < 4:
        continue
    cfg_str, layer_str, slug = parts[-1], parts[-2], parts[-3]
    proj_key = "/".join(parts[:-3])
    if not cfg_str.startswith("CFG") or not layer_str.startswith("L"):
        continue
    mv = rec.get("mean_pixel_var")
    if mv is None:
        continue
    cfg = float(cfg_str[3:])
    layer = int(layer_str[1:])
    sv_groups[(proj_key, cfg)][slug][layer] = mv

# Plot: one subplot per (proj_key, cfg), lines coloured by category
for (proj_key, cfg), slug_data in sorted(sv_groups.items()):
    slug_data = {s: d for s, d in slug_data.items() if len(d) >= 3}
    if not slug_data:
        continue

    fig, ax = plt.subplots(figsize=(12, 5))
    conv_layers: dict[str, dict] = {}

    for slug, layer_vars in sorted(slug_data.items()):
        xs = sorted(layer_vars)
        ys = [layer_vars[x] for x in xs]
        cat = SLUG_TO_CAT.get(slug, "unknown")
        color = CAT_COLORS.get(cat, "#888")
        ax.plot(xs, ys, marker="o", ms=3, lw=1.4, color=color, label=f"{slug[:30]} ({cat})")
        conv_layer = xs[int(np.argmin(ys))]
        ax.axvline(conv_layer, color=color, lw=0.5, alpha=0.3)
        conv_layers[slug] = {"layer": conv_layer, "min_var": min(ys), "cat": cat}

    ax.set_xlabel("Layer index")
    ax.set_ylabel("Mean pixel variance (across seeds)")
    ax.set_title(f"Convergence Layer — {proj_key} | CFG={cfg:g}")
    ax.legend(fontsize=7, loc="upper right", ncol=2)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Convergence layer summary per category
    print(f"\n{proj_key} | CFG={cfg:g} — convergence layers:")
    for slug, info in sorted(conv_layers.items(), key=lambda x: x[1]["cat"]):
        print(f"  L{info['layer']:2d}  {info['cat']:<20}  min_var={info['min_var']:.2f}  {slug[:40]}")

## 10. Exp 5 analysis — D12 test: variance profile by category

Are periphery probe variance profiles qualitatively different from benchmark probes?

Two diagnostics:
1. **Distribution of convergence layers per category** — does periphery peak at a different layer?
2. **Variance range (max − min) per probe** — a flat profile indicates weak conditioning regardless of layer; a sharp dip indicates a well-defined convergence point.

In [ ]:
# Load convergence_summary.json if analyze phase has run
conv_summary_path = RESULTS_DIR / "analysis" / "convergence_summary.json"
if conv_summary_path.exists():
    with open(conv_summary_path) as f:
        conv_summary = json.load(f)

    # Collect per-category convergence layers and variance ranges
    cat_conv_layers: dict[str, list[int]] = defaultdict(list)
    cat_var_ranges:  dict[str, list[float]] = defaultdict(list)

    for proj_key, cfg_data in conv_summary.items():
        for cfg_str, slug_data in cfg_data.items():
            for slug, info in slug_data.items():
                cat = SLUG_TO_CAT.get(slug, "unknown")
                cat_conv_layers[cat].append(info["convergence_layer"])
                cat_var_ranges[cat].append(info["max_pixel_var"] - info["min_pixel_var"])

    cats = sorted(cat_conv_layers)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

    # Convergence layer distribution
    for cat in cats:
        ax1.scatter(
            [cat] * len(cat_conv_layers[cat]),
            cat_conv_layers[cat],
            color=CAT_COLORS.get(cat, "#888"),
            alpha=0.7, s=60,
        )
    ax1.set_xlabel("Category")
    ax1.set_ylabel("Convergence layer index")
    ax1.set_title("Convergence Layer by Category (D12 test)")
    ax1.grid(True, alpha=0.3, axis="y")

    # Variance range (sharpness of convergence dip)
    for cat in cats:
        ax2.scatter(
            [cat] * len(cat_var_ranges[cat]),
            cat_var_ranges[cat],
            color=CAT_COLORS.get(cat, "#888"),
            alpha=0.7, s=60,
        )
    ax2.set_xlabel("Category")
    ax2.set_ylabel("max − min pixel variance")
    ax2.set_title("Variance Range by Category\n(high = sharp convergence dip)")
    ax2.grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    plt.show()

    # Numeric summary
    print("Category means:")
    print(f"{'Category':<22} {'mean conv. layer':>16} {'mean var range':>14} {'n':>4}")
    for cat in cats:
        cl = cat_conv_layers[cat]
        vr = cat_var_ranges[cat]
        print(f"{cat:<22} {np.mean(cl):>16.1f} {np.mean(vr):>14.2f} {len(cl):>4}")
else:
    print("convergence_summary.json not found — run the analyze phase first.")

## 11. Exp 5 analysis — alpha compression by layer

`cosine_distance(proj_α1(act), proj_α1000(act))` at each layer, for each probe.  
This is the layer-sweep extension of the Exp 1/3 last-layer-only result.

Stored in `manifest["probe_clip_vectors"][proj_key][slug][layer_idx]`.

In [ ]:
from scipy.spatial.distance import cosine as cosine_dist

alpha1_key    = "per_layer_alpha1"
alpha1000_key = "per_layer_alpha1000"

vecs_a1    = clip_vecs.get(alpha1_key, {})
vecs_a1000 = clip_vecs.get(alpha1000_key, {})

if not vecs_a1 or not vecs_a1000:
    print("probe_clip_vectors not yet populated — run generate phase first.")
else:
    # Compute cosine distance per (slug, layer)
    common_slugs = sorted(set(vecs_a1) & set(vecs_a1000))

    fig, ax = plt.subplots(figsize=(12, 5))
    for slug in common_slugs:
        layers_a1    = vecs_a1[slug]
        layers_a1000 = vecs_a1000[slug]
        common_layers = sorted(set(layers_a1) & set(layers_a1000), key=int)
        if not common_layers:
            continue
        xs = [int(l) for l in common_layers]
        ys = [
            cosine_dist(np.array(layers_a1[l]), np.array(layers_a1000[l]))
            for l in common_layers
        ]
        cat = SLUG_TO_CAT.get(slug, "unknown")
        ax.plot(xs, ys, marker="o", ms=3, lw=1.4,
                color=CAT_COLORS.get(cat, "#888"),
                label=f"{slug[:28]} ({cat})")

    ax.set_xlabel("Layer index")
    ax.set_ylabel("Cosine distance (α=1 vs α=1000)")
    ax.set_title("Alpha Compression Sensitivity by Layer — Pythia-410m")
    ax.legend(fontsize=7, loc="upper left", ncol=2)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 12. Exp 6 analysis — perturbation family CLIP coherence

Pairwise cosine distance matrix for all perturbation probes at L23, α=1.  
Within-family distances (same `group_id`) should be smaller than between-family distances if the projection is sensitive to semantics rather than surface form.

In [ ]:
layer_str = "23"
proj_key  = alpha1_key

# Collect CLIP vectors for all Exp 6 probes
exp6_slugs:  list[str] = []
exp6_groups: list[str] = []
exp6_vtype:  list[str] = []
exp6_vecs:   list[np.ndarray] = []

slug_vecs = clip_vecs.get(proj_key, {})
if not slug_vecs:
    print("probe_clip_vectors not yet populated — run generate phase first.")
else:
    for gid, variant_texts in EXP6_PROBE_TEXTS.items():
        for vtype, text in zip(VARIANT_ORDER, variant_texts):
            slug = _slug(text)
            vec_dict = slug_vecs.get(slug, {})
            vec = vec_dict.get(layer_str)
            if vec is None:
                print(f"  missing: {gid}/{vtype} ({slug})")
                continue
            exp6_slugs.append(slug)
            exp6_groups.append(gid)
            exp6_vtype.append(vtype)
            exp6_vecs.append(np.array(vec))

    n = len(exp6_vecs)
    print(f"{n} probes loaded for Exp 6 pairwise analysis")

    if n > 0:
        # Compute pairwise cosine distance matrix
        D = np.zeros((n, n))
        for i in range(n):
            for j in range(n):
                D[i, j] = cosine_dist(exp6_vecs[i], exp6_vecs[j]) if i != j else 0.0

        # Plot
        labels = [f"{g.split('_')[-1]}/{v[:4]}" for g, v in zip(exp6_groups, exp6_vtype)]
        group_boundaries = []
        prev = None
        for i, g in enumerate(exp6_groups):
            if g != prev:
                group_boundaries.append(i)
                prev = g

        fig, ax = plt.subplots(figsize=(9, 8))
        im = ax.imshow(D, cmap="viridis", vmin=0)
        fig.colorbar(im, ax=ax, label="Cosine distance")
        ax.set_xticks(range(n)); ax.set_xticklabels(labels, rotation=90, fontsize=8)
        ax.set_yticks(range(n)); ax.set_yticklabels(labels, fontsize=8)
        for b in group_boundaries:
            ax.axhline(b - 0.5, color="white", lw=1.5)
            ax.axvline(b - 0.5, color="white", lw=1.5)
        ax.set_title("Perturbation Family CLIP Cosine Distances (L23, α=1)\nWhite lines = family boundaries")
        plt.tight_layout()
        plt.show()

        # Within- vs between-family statistics
        within, between = [], []
        for i in range(n):
            for j in range(i + 1, n):
                (within if exp6_groups[i] == exp6_groups[j] else between).append(D[i, j])

        print(f"Within-family  distances: mean={np.mean(within):.4f}  std={np.std(within):.4f}  n={len(within)}")
        print(f"Between-family distances: mean={np.mean(between):.4f}  std={np.std(between):.4f}  n={len(between)}")
        ratio = np.mean(between) / np.mean(within) if np.mean(within) > 0 else float("inf")
        print(f"Between/within ratio: {ratio:.2f}  (>1 = surface variants cluster within family)")

## 13. Exp 7 — LPIPS sensitivity at CFG=25 (deferred)

Run once Exp 5 images exist. Re-run generate with `track_lpips: True` to compute LPIPS inline,  
or run the `metrics` phase post-hoc.

Key question: does r(LPIPS, cosine_dist) exceed the noise floor at CFG=25?  
Expected: Pythia's typical cosine_dist ≈ 0.07 should now exceed δ at CFG=25 (vs. CFG=7.5 in Exp 1/3 where it didn't).

```bash
# When ready:
python run_experiment.py --config experiment_config_exp5_layersweep.json --phase metrics
```

## 14. Summary

| Experiment | Status | Primary metric | Key question |
|------------|--------|----------------|--------------|
| **Exp 5** layer sweep | ☐ pending | `mean_pixel_var` convergence layer | Is convergence layer consistent across probe categories? D12: does periphery differ? |
| **Exp 6** perturbation families | ☐ pending | CLIP cosine distance within/between family | Do surface variants cluster within family in CLIP space? |
| **Exp 7** LPIPS at CFG=25 | ☐ deferred | r(LPIPS, cosine_dist) | Is LPIPS now usable as secondary metric? |